In [1]:

import os
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
import json
import requests


In [2]:
load_dotenv()

True

##STEP-1: TEST NEO4J CONNECTION-Run Neo4j locally with Docker (ports 7474 and 7687) or on Aura, and connect from Python with the neo4j driver (verify_connectivity).

In [3]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [4]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("Connected to Neo4j database Successfully")

Connected to Neo4j database Successfully


In [5]:
driver.verify_connectivity()
print("Connection to Neo4j database verified successfully")

Connection to Neo4j database verified successfully


In [6]:
"""
Step 2: build the SENTINEL fraud graph in Neo4j from my own Supabase load.

    Supabase (Project Work 5: sql/create_table + python ingest.py all)
        transactions, labels, customers, devices, merchants
            |
            v   read with the Supabase REST API (1,000 rows per request)
        pandas: join fraud_label onto each transaction
            |
            v   MERGE in batches of 1,000
        Neo4j:
        (:Customer)-[:USED_DEVICE]->(:Device)
        (:Customer)-[:PAID {amount, time, fraud_label}]->(:Merchant)

Run:  python step2_supabase_to_neo4j.py
Needs in .env:  NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, SUPABASE_URL, SUPABASE_KEY
Optional:       SAMPLE_SIZE (default 10000, must be at least 5000)
"""

'\nStep 2: build the SENTINEL fraud graph in Neo4j from my own Supabase load.\n\n    Supabase (Project Work 5: sql/create_table + python ingest.py all)\n        transactions, labels, customers, devices, merchants\n            |\n            v   read with the Supabase REST API (1,000 rows per request)\n        pandas: join fraud_label onto each transaction\n            |\n            v   MERGE in batches of 1,000\n        Neo4j:\n        (:Customer)-[:USED_DEVICE]->(:Device)\n        (:Customer)-[:PAID {amount, time, fraud_label}]->(:Merchant)\n\nRun:  python step2_supabase_to_neo4j.py\nNeeds in .env:  NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, SUPABASE_URL, SUPABASE_KEY\nOptional:       SAMPLE_SIZE (default 10000, must be at least 5000)\n'

In [31]:
# 0. Settings
# ---------------------------------------------------------------------------
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

SUPABASE_URL = os.environ["SUPABASE_URL"].rstrip("/")   # https://<project>.supabase.co
SUPABASE_KEY = os.environ["SUPABASE_KEY"]                # Project Settings > API > anon key
SAMPLE_SIZE = int(os.getenv("SAMPLE_SIZE", "10000"))     # the task asks for at least 5,000

PAGE = 1000    # Supabase returns at most 1,000 rows per request
BATCH = 1000   # rows sent to Neo4j per query

REST = f"{SUPABASE_URL}/rest/v1/"
HEADERS = {"apikey": SUPABASE_KEY, "Authorization": f"Bearer {SUPABASE_KEY}"}

In [32]:

# ---------------------------------------------------------------------------
# 1. Read from Supabase
# ---------------------------------------------------------------------------
def supabase_get(table, params):
    """One GET request to the Supabase REST API. Returns a list of rows (dicts)."""
    r = requests.get(REST + table, headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()  # stops with a clear error if the URL, key or table name is wrong
    return r.json()


def read_transactions(n):
    """Read the first n transactions, ordered by transaction_id, one page of 1,000 at a time."""
    rows = []
    while len(rows) < n:
        page = supabase_get("transactions", {
            "select": "transaction_id,customer_id,merchant_id,device_id,"
                      "transaction_amount,payment_channel,transaction_time",
            "order": "transaction_id",
            "limit": min(PAGE, n - len(rows)),
            "offset": len(rows),
        })
        if not page:          # no more rows in the table
            break
        rows += page
        print(f"  transactions read: {len(rows):,}")
    return pd.DataFrame(rows)


def read_by_ids(table, id_column, ids, select="*"):
    """Read only the rows whose id is in `ids` (200 ids per request so the URL stays short)."""
    ids = list(dict.fromkeys(str(x) for x in ids if pd.notna(x)))  # unique, no blanks, keeps order
    rows = []
    for i in range(0, len(ids), 200):
        chunk = ",".join(ids[i:i + 200])
        rows += supabase_get(table, {"select": select, id_column: f"in.({chunk})"})
    return pd.DataFrame(rows)


print("Reading from Supabase:", SUPABASE_URL)
tx = read_transactions(SAMPLE_SIZE)
labels = read_by_ids("labels", "transaction_id", tx["transaction_id"], "transaction_id,fraud_label")
customers = read_by_ids("customers", "customer_id", tx["customer_id"])
devices = read_by_ids("devices", "device_id", tx["device_id"])
merchants = read_by_ids("merchants", "merchant_id", tx["merchant_id"])

Reading from Supabase: https://xaxjeklyoquiqkiutwnf.supabase.co
  transactions read: 1,000
  transactions read: 2,000
  transactions read: 3,000
  transactions read: 4,000
  transactions read: 5,000
  transactions read: 6,000
  transactions read: 7,000
  transactions read: 8,000
  transactions read: 9,000
  transactions read: 10,000


In [33]:

# ---------------------------------------------------------------------------
# 2. Prepare the data with pandas
# ---------------------------------------------------------------------------
# fraud_label lives in the labels table: join it on transaction_id (missing label = 0, genuine)
tx = tx.merge(labels, on="transaction_id", how="left")
tx["fraud_label"] = tx["fraud_label"].fillna(0).astype(int)
tx["transaction_time"] = pd.to_datetime(tx["transaction_time"], utc=True, format="ISO8601")
tx = tx.sort_values("transaction_time")

print(f"\nSample: {len(tx):,} transactions | {len(customers):,} customers | {len(devices):,} devices | "
      f"{len(merchants):,} merchants | {tx['fraud_label'].sum():,} fraud | "
      f"{tx['transaction_time'].min():%Y-%m-%d} to {tx['transaction_time'].max():%Y-%m-%d}")
if len(tx) < 5000:
    raise SystemExit("Fewer than 5,000 transactions: check that ingest.py all loaded the transactions table.")


def city(location_json):
    """merchant_location is stored as JSON text like {"city":"Delhi","lat":..,"lon":..}."""
    try:
        return json.loads(location_json).get("city")
    except (TypeError, ValueError):
        return location_json.get("city") if isinstance(location_json, dict) else None


customer_rows = [{"id": r["customer_id"], "age": int(r["customer_age"]),
                  "city": r.get("home_city"), "region": r.get("region")}
                 for r in customers.to_dict("records")]

device_rows = [{"id": r["device_id"], "kind": r.get("device_kind")}
               for r in devices.to_dict("records")]

merchant_rows = [{"id": r["merchant_id"], "name": r.get("synthetic_name"),
                  "category": r.get("merchant_category"), "city": city(r.get("merchant_location"))}
                 for r in merchants.to_dict("records")]

# USED_DEVICE: one edge per (customer, device), with every time that device was used
used = (tx.groupby(["customer_id", "device_id"])["transaction_time"]
          .apply(lambda s: [t.isoformat() for t in s])
          .reset_index(name="times"))
used_rows = used.to_dict("records")

# PAID: one edge per transaction, with amount, time and fraud_label
paid_rows = [{"tx": r["transaction_id"], "c": r["customer_id"], "m": r["merchant_id"], "d": r["device_id"] if pd.notna(r["device_id"]) else None,
              "amount": float(r["transaction_amount"]), "time": r["transaction_time"].isoformat(),
              "fraud": int(r["fraud_label"]), "channel": r["payment_channel"]}
             for r in tx.to_dict("records")]


Sample: 10,000 transactions | 2,001 customers | 2,334 devices | 200 merchants | 60 fraud | 2024-09-30 to 2024-12-31


In [34]:

# ---------------------------------------------------------------------------
# 3. Connect to Neo4j
# ---------------------------------------------------------------------------
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("\nConnected to Neo4j:", NEO4J_URI)


def run(query, **params):
    records, _, _ = driver.execute_query(query, params)
    return [r.data() for r in records]


def write_in_batches(query, rows, name):
    for i in range(0, len(rows), BATCH):
        run(query, rows=rows[i:i + BATCH])
    print(f"  {name}: {len(rows):,} rows merged")



Connected to Neo4j: neo4j://localhost:7687


In [35]:

# ---------------------------------------------------------------------------
# 4. Uniqueness constraints (also create the indexes MERGE uses to find ids fast)
# ---------------------------------------------------------------------------
run("CREATE CONSTRAINT customer_id IF NOT EXISTS FOR (c:Customer) REQUIRE c.customer_id IS UNIQUE")
run("CREATE CONSTRAINT device_id IF NOT EXISTS FOR (d:Device) REQUIRE d.device_id IS UNIQUE")
run("CREATE CONSTRAINT merchant_id IF NOT EXISTS FOR (m:Merchant) REQUIRE m.merchant_id IS UNIQUE")
run("CREATE CONSTRAINT paid_tx_id IF NOT EXISTS FOR ()-[p:PAID]-() REQUIRE p.transaction_id IS UNIQUE")


[]

In [36]:
# ---------------------------------------------------------------------------
# 5. Nodes: MERGE on the id, then SET the other properties
# ---------------------------------------------------------------------------
print("\nLoading nodes")
write_in_batches("""
UNWIND $rows AS row
MERGE (c:Customer {customer_id: row.id})
SET c.customer_age = row.age, c.home_city = row.city, c.region = row.region
""", customer_rows, "Customer")

write_in_batches("""
UNWIND $rows AS row
MERGE (d:Device {device_id: row.id})
SET d.device_kind = row.kind
""", device_rows, "Device")

write_in_batches("""
UNWIND $rows AS row
MERGE (m:Merchant {merchant_id: row.id})
SET m.name = row.name, m.merchant_category = row.category, m.city = row.city
""", merchant_rows, "Merchant")



Loading nodes
  Customer: 2,001 rows merged
  Device: 2,334 rows merged
  Merchant: 200 rows merged


In [37]:
# ---------------------------------------------------------------------------
# 6. Relationships
# ---------------------------------------------------------------------------
print("\nLoading relationships")
write_in_batches("""
UNWIND $rows AS row
MATCH (c:Customer {customer_id: row.customer_id})
MATCH (d:Device   {device_id:   row.device_id})
MERGE (c)-[u:USED_DEVICE]->(d)
SET u.times      = [t IN row.times | datetime(t)],
    u.first_seen = datetime(row.times[0]),
    u.last_seen  = datetime(row.times[-1]),
    u.tx_count   = size(row.times)
""", used_rows, "USED_DEVICE")

write_in_batches("""
UNWIND $rows AS row
MATCH (c:Customer {customer_id: row.c})
MATCH (m:Merchant {merchant_id: row.m})
MERGE (c)-[p:PAID {transaction_id: row.tx}]->(m)
SET p.amount          = row.amount,
    p.time            = datetime(row.time),
    p.fraud_label     = row.fraud,
    p.payment_channel = row.channel,
    p.device_id       = row.d
""", paid_rows, "PAID")


Loading relationships
  USED_DEVICE: 2,363 rows merged
  PAID: 10,000 rows merged


In [38]:
# ---------------------------------------------------------------------------
# 7. Check what is in the graph
# ---------------------------------------------------------------------------
print("\nGraph now contains:")
for label in ["Customer", "Device", "Merchant"]:
    print(f"  (:{label}) nodes:", run(f"MATCH (n:{label}) RETURN count(n) AS n")[0]["n"])
for rel in ["USED_DEVICE", "PAID"]:
    print(f"  [:{rel}] edges:", run(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS n")[0]["n"])
print("  fraud PAID edges:", run("MATCH ()-[p:PAID {fraud_label: 1}]->() RETURN count(p) AS n")[0]["n"])

print("\nSample PAID edges:")
for row in run("""
MATCH (c:Customer)-[p:PAID]->(m:Merchant)
RETURN c.customer_id AS customer, m.merchant_category AS category,
       p.amount AS amount, toString(p.time) AS time, p.fraud_label AS fraud_label
ORDER BY p.fraud_label DESC, p.time LIMIT 5"""):
    print(" ", row)


print("\nDone. Run it again: the counts stay the same (MERGE, no duplicates).")


Graph now contains:
  (:Customer) nodes: 2001
  (:Device) nodes: 2334
  (:Merchant) nodes: 200
  [:USED_DEVICE] edges: 2363
  [:PAID] edges: 10000
  fraud PAID edges: 60

Sample PAID edges:
  {'customer': 'C_bf9ac8d42105c1ff', 'category': 'electronics', 'amount': 11553.43, 'time': '2024-10-01T18:56:26.182Z', 'fraud_label': 1}
  {'customer': 'C_134f0dafd6bef51e', 'category': 'gaming', 'amount': 38804.13, 'time': '2024-10-04T20:38:12.223Z', 'fraud_label': 1}
  {'customer': 'C_052085910a76325d', 'category': 'gift_cards', 'amount': 6728.57, 'time': '2024-10-05T14:05:44.56Z', 'fraud_label': 1}
  {'customer': 'C_79f0d8ce6622d6bf', 'category': 'gift_cards', 'amount': 9347.48, 'time': '2024-10-05T21:50:11.577Z', 'fraud_label': 1}
  {'customer': 'C_f9b5b518b79a7647', 'category': 'gift_cards', 'amount': 10060.58, 'time': '2024-10-07T23:10:06.294Z', 'fraud_label': 1}

Done. Run it again: the counts stay the same (MERGE, no duplicates).


# Step 4: Fraud queries in Cypher
The same queries are in `queries.cypher` so you can paste them into Neo4j Browser.

### 4a. Devices shared by 3 or more customers within 7 days
For each device, every use is tried as the start of a 7-day window, and the window with the
most distinct customers is kept.

In [39]:
Q_SHARED_DEVICES = """
MATCH (d:Device)<-[:USED_DEVICE]-(:Customer)
WITH d, count(*) AS total_customers
WHERE total_customers >= $min_customers
MATCH (d)<-[u:USED_DEVICE]-(c:Customer)
UNWIND u.times AS t
WITH d, collect({customer: c.customer_id, time: t}) AS uses
UNWIND uses AS start
WITH d, start.time AS window_start,
     [x IN uses WHERE x.time >= start.time AND x.time < start.time + duration({days: $days}) | x.customer] AS in_window
UNWIND in_window AS customer
WITH d, window_start, collect(DISTINCT customer) AS customers
WHERE size(customers) >= $min_customers
WITH d, window_start, customers
ORDER BY size(customers) DESC, window_start
WITH d, collect({start: window_start, customers: customers})[0] AS best
RETURN d.device_id AS device_id, d.device_kind AS device_kind,
       size(best.customers) AS customers_in_window,
       toString(date(best.start)) AS window_start, best.customers AS customer_ids
ORDER BY customers_in_window DESC, window_start
LIMIT 25
"""
shared = pd.DataFrame(run(Q_SHARED_DEVICES, min_customers=3, days=7))
print(f"{len(shared)} devices shared by 3+ customers within 7 days")
print(shared.to_string() if len(shared) else "(none in this sample: try a bigger NUM_PARTS)")

1 devices shared by 3+ customers within 7 days
            device_id device_kind  customers_in_window window_start                                                  customer_ids
0  D_0c947fb45bb50e72      mobile                    3   2024-10-25  [C_984cb84b0a294365, C_148927ad36a84923, C_1aa451e90647f952]


### 4b. Customers within 2 hops of a fraud-labelled payment
A fraud payment is a `PAID` edge with `fraud_label = 1`. Starting from the customer who made it,
1 to 2 hops over `USED_DEVICE` / `PAID` reaches the customers who share a device with them
(customer -> device -> customer) or paid the same merchant (customer -> merchant -> customer).

In [ ]:
Q_NEAR_FRAUD = """
MATCH (fc:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
WITH DISTINCT fc
MATCH (fc)-[r:USED_DEVICE|PAID]-(link)-[r2:USED_DEVICE|PAID]-(c:Customer)
WHERE c <> fc
WITH c, fc, link
RETURN c.customer_id AS customer_id,
       count(DISTINCT fc) AS fraud_customers_nearby,
       count(DISTINCT CASE WHEN link:Device THEN link END) AS shared_devices,
       count(DISTINCT CASE WHEN link:Merchant THEN link END) AS shared_merchants,
       EXISTS { (c)-[:PAID {fraud_label: 1}]->() } AS has_own_fraud
ORDER BY shared_devices DESC, fraud_customers_nearby DESC, shared_merchants DESC
LIMIT 25
"""
near = pd.DataFrame(run(Q_NEAR_FRAUD))
total_near = run("""
MATCH (fc:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
WITH DISTINCT fc
MATCH (fc)-[:USED_DEVICE|PAID]-(link)-[:USED_DEVICE|PAID]-(c:Customer)
WHERE c <> fc
RETURN count(DISTINCT c) AS all_customers,
       count(DISTINCT CASE WHEN link:Device THEN c END) AS via_shared_device
""")[0]
print("customers within 2 hops of a fraud payment:", total_near)
print(near.to_string())

### 4c. Merchants most often paid from shared devices
A shared device is one used by 2 or more customers. Each `PAID` edge stores the `device_id` it was made
from, so we count the payments made from a shared device, per merchant.

In [40]:
Q_SHARED_DEVICE_MERCHANTS = """
MATCH (d:Device)<-[:USED_DEVICE]-(:Customer)
WITH d, count(*) AS n_customers
WHERE n_customers >= $min_customers
MATCH (d)<-[:USED_DEVICE]-(c:Customer)-[p:PAID]->(m:Merchant)
WHERE p.device_id = d.device_id
RETURN m.merchant_id AS merchant_id, m.name AS merchant, m.merchant_category AS category,
       count(p) AS payments_from_shared_devices,
       count(DISTINCT d) AS shared_devices,
       count(DISTINCT c) AS customers,
       sum(p.fraud_label) AS fraud_payments,
       round(sum(p.amount), 2) AS total_amount
ORDER BY payments_from_shared_devices DESC, fraud_payments DESC
LIMIT 10
"""
top_merchants = pd.DataFrame(run(Q_SHARED_DEVICE_MERCHANTS, min_customers=2))
print(top_merchants.to_string())

          merchant_id                   merchant     category  payments_from_shared_devices  shared_devices  customers  fraud_payments  total_amount
0  M_9dd7da301c13c892  Synthetic Merchant 000146   gift_cards                            28               4          4               0     353281.39
1  M_ce0b45ab7127ca6a  Synthetic Merchant 000014  electronics                            17               2          2               1     273931.62
2  M_06868839e93b9a80  Synthetic Merchant 000026   gift_cards                            16               2          2               0      73384.76
3  M_cfe75c4398440f99  Synthetic Merchant 000176   gift_cards                            16               2          2               0      77599.68
4  M_fb626fe084cf053e  Synthetic Merchant 000166   gift_cards                            14               2          2               1     244775.40
5  M_096150fc8e36724b  Synthetic Merchant 000005       gaming                            14               

## Step 5: Pick one suspicious cluster for the screenshot
The most suspicious device is the one shared by the most customers in 7 days (step 4a); if none,
the device shared by the most customers that also has a fraud payment.
This cell prints the Cypher to paste into Neo4j Browser (http://localhost:7474) 

In [41]:
from pathlib import Path
cluster = run("""
MATCH (d:Device)<-[u:USED_DEVICE]-(c:Customer)
WITH d, count(c) AS n,
     sum(size([(c)-[p:PAID {fraud_label: 1}]->() WHERE p.device_id = d.device_id | p])) AS fraud
WHERE n >= 2
RETURN d.device_id AS device_id
ORDER BY n DESC, fraud DESC
LIMIT 1
""")
device_id = shared["device_id"].iloc[0] if len(shared) else (cluster[0]["device_id"] if cluster else None)

if device_id is None:
    print("no shared device in this sample")
else:
    browser_query = f"""MATCH (d:Device {{device_id: "{device_id}"}})<-[u:USED_DEVICE]-(c:Customer)
OPTIONAL MATCH (c)-[p:PAID]->(m:Merchant)
WHERE p.device_id = d.device_id
RETURN d, u, c, p, m"""
    print("Paste this into Neo4j Browser:\n")
    print(browser_query)

    people = pd.DataFrame(run("""
    MATCH (d:Device {device_id: $id})<-[u:USED_DEVICE]-(c:Customer)
    RETURN c.customer_id AS customer, c.customer_age AS age, c.home_city AS home_city,
           toString(u.first_seen) AS first_used, toString(u.last_seen) AS last_used, u.tx_count AS uses
    ORDER BY first_used""", id=device_id))
    pays = pd.DataFrame(run("""
    MATCH (d:Device {device_id: $id})<-[:USED_DEVICE]-(c:Customer)-[p:PAID]->(m:Merchant)
    WHERE p.device_id = d.device_id
    RETURN c.customer_id AS customer, m.name AS merchant, m.merchant_category AS category,
           m.city AS merchant_city, p.amount AS amount, toString(p.time) AS time, p.fraud_label AS fraud_label
    ORDER BY time""", id=device_id))
    kind = run("MATCH (d:Device {device_id: $id}) RETURN d.device_kind AS k", id=device_id)[0]["k"]
    print(f"\nDevice {device_id} ({kind}): {len(people)} customers, {len(pays)} payments, "
          f"{int(pays['fraud_label'].sum())} fraud, total {pays['amount'].sum():,.2f}")
    print(people.to_string())
    print(pays.to_string())

    report = [f"# Suspicious cluster: device {device_id} ({kind})", "",
              "## Cypher used for the screenshot", "```cypher", browser_query, "```", "",
              f"- Customers on this device: **{len(people)}**",
              f"- Payments made from it: **{len(pays)}**, total **{pays['amount'].sum():,.2f}**",
              f"- Fraud-labelled payments: **{int(pays['fraud_label'].sum())}**",
              f"- Time span: {pays['time'].min()[:16]} to {pays['time'].max()[:16]}",
              f"- Merchant categories: {', '.join(pays['category'].value_counts().index)}",
              f"- Customers' home cities: {', '.join(sorted(set(people['home_city'])))}", "",
              "## Customers", people.to_markdown(index=False), "",
              "## Payments from this device", pays.to_markdown(index=False), ""]
    Path("cluster_report.md").write_text("\n".join(report), encoding="utf-8")
    print("\nwrote cluster_report.md")

Paste this into Neo4j Browser:

MATCH (d:Device {device_id: "D_0c947fb45bb50e72"})<-[u:USED_DEVICE]-(c:Customer)
OPTIONAL MATCH (c)-[p:PAID]->(m:Merchant)
WHERE p.device_id = d.device_id
RETURN d, u, c, p, m

Device D_0c947fb45bb50e72 (mobile): 3 customers, 19 payments, 1 fraud, total 262,206.51
             customer  age home_city                first_used                 last_used  uses
0  C_1aa451e90647f952   48     Delhi  2024-10-25T23:05:32.631Z  2024-10-25T23:12:37.994Z     6
1  C_984cb84b0a294365   43   Lucknow   2024-10-27T06:40:22.83Z  2024-10-27T06:46:58.291Z     7
2  C_148927ad36a84923   28     Patna  2024-10-29T19:43:34.353Z  2024-10-29T19:48:34.771Z     6
              customer                   merchant    category merchant_city     amount                      time  fraud_label
0   C_1aa451e90647f952  Synthetic Merchant 000185      gaming       Lucknow    4088.01  2024-10-25T23:05:32.631Z            0
1   C_1aa451e90647f952  Synthetic Merchant 000185      gaming       Luc

In [ ]:
driver.close()